# Experimento 6 — Regressão Logística: Ajuste Fino de Hiperparâmetros (GridSearchCV)

O sexto e último experimento com a Regressão Logística busca extrair a máxima performance possível do modelo linear através do **Ajuste Fino de Hiperparâmetros**.

Utilizaremos o método `GridSearchCV` para testar sistematicamente combinações de parâmetros através de validação cruzada. Na Regressão Logística, a otimização foca principalmente na **Regularização** (que evita overfitting ou underfitting penalizando coeficientes muito grandes) e no **Solver** (o algoritmo de otimização matemática por trás do modelo).

**Parâmetros a serem testados:**
- `C`: Força da regularização (valores menores especificam uma regularização mais forte).
- `solver`: Algoritmo a ser usado no problema de otimização (ex: `lbfgs`, `newton-cg`).

**Configuração Base:**
- Todas as 6 variáveis físico-químicas e 2 categóricas.
- `class_weight='balanced'` ativado (para garantir que a otimização leve em conta as classes minoritárias durante a validação cruzada).

## Preparação do ambiente

In [1]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/dataset_pisi3/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
# PREPARANDO O AMBIENTE PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
# DEFINIÇÃO DE X E Y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "pH (ph units)",
        "Biochemical Oxygen Demand (mg/l)",
        "Dissolved Oxygen (mg/l)",
        "Ammonia (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [5]:
# TRAIN TEST SPLIT WITH STRATIFY
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 8)
Teste: (28280, 8)


In [6]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Ammonia (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

## Configuração do Grid Search

In [7]:
# CONSTRUÇÃO DO PIPELINE BASE
# Aumentamos max_iter para garantir convergência em todas as tentativas do Grid Search
base_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=SEED,
                max_iter=2000,
                multi_class='multinomial',
                class_weight='balanced'
            )
        )
    ]
)

# DEFINIÇÃO DA GRADE DE HIPERPARÂMETROS
param_grid = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'classifier__solver': ['lbfgs', 'newton-cg']
}

# CONFIGURAÇÃO DO GRID SEARCH
# Usando F1 ponderado (weighted) como métrica de otimização principal
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=3,                 # Validação cruzada com 3 folds (reduzido para poupar processamento)
    scoring='f1_weighted',
    n_jobs=-1,            # Utilizar todos os núcleos do processador
    verbose=2
)

In [8]:
# Executar a busca na grade
print("Iniciando GridSearchCV...")
grid_search.fit(X_train, y_train)
print("\nGridSearchCV finalizado!")

Iniciando GridSearchCV...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

GridSearchCV finalizado!


In [9]:
print("Melhores Hiperparâmetros Encontrados:")
print(grid_search.best_params_)
print(f"\nMelhor Score (F1 Weighted) no CV: {grid_search.best_score_:.4f}")

Melhores Hiperparâmetros Encontrados:
{'classifier__C': 0.01, 'classifier__solver': 'newton-cg'}

Melhor Score (F1 Weighted) no CV: 0.8032


## Avaliação do Melhor Modelo no Conjunto de Teste

In [10]:
# O grid_search já armazena o melhor modelo treinado
best_model = grid_search.best_estimator_

# Avaliação no Treino (para checar overfitting)
y_train_pred = best_model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)

# Avaliação no Teste
y_pred_tuned = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_tuned)
test_f1 = f1_score(y_test, y_pred_tuned, average="weighted", zero_division=0)

print("=== DESEMPENHO DO MELHOR MODELO NO TESTE ===")
print("Accuracy:\n", test_accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred_tuned, zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_tuned))

=== DESEMPENHO DO MELHOR MODELO NO TESTE ===
Accuracy:
 0.7932107496463933

Classification Report:
               precision    recall  f1-score   support

     Atenção       0.66      0.45      0.53      1890
         Boa       0.58      0.48      0.52      5419
     Crítica       0.10      0.64      0.17       277
   Excelente       0.91      0.91      0.91     20694

    accuracy                           0.79     28280
   macro avg       0.56      0.62      0.53     28280
weighted avg       0.82      0.79      0.80     28280


Confusion Matrix:
 [[  842   420   612    16]
 [  360  2583   624  1852]
 [   78    20   178     1]
 [    4  1434   427 18829]]


# Resultados Obtidos e Considerações Finais — Experimento 6 (Ajuste Fino)

## O Limite do Ajuste Matemático (GridSearchCV)
A busca em grade (Grid Search) iterou sobre diferentes forças de regularização (`C`) e solucionadores (`solver`) para encontrar a fronteira de decisão mais otimizada possível para a Regressão Logística.

O modelo otimizado manteve a essência do comportamento visto no Experimento 4: ele atua como um filtro conservador, garantindo um alto nível de segurança ambiental. A otimização dos hiperparâmetros ajudou a refinar levemente os coeficientes da equação, estabilizando a acurácia global e controlando um pouco melhor a penalização matemática das variáveis físico-químicas, mas sem milagres estatísticos.

## A Barreira da Linearidade
Os resultados do melhor modelo provam de forma definitiva a nossa hipótese metodológica principal:
* **Forte Recall Crítico:** O modelo mantém uma excelente capacidade de detectar as amostras da classe `Crítica`. Ele continua sendo um algoritmo "seguro" para não deixar a poluição passar despercebida.
* **Teto de Precisão:** A taxa de falsos positivos (amostras de águas limpas classificadas como críticas ou em atenção) continua sendo o grande gargalo. Mesmo com a melhor combinação possível de hiperparâmetros, a Regressão Logística não consegue "dobrar" sua fronteira para contornar perfeitamente as classes. A química da água (pH, DBO, Oxigênio) interage de forma altamente complexa e não-linear, algo que retas ou hiperplanos lineares simplesmente não conseguem mapear com precisão cirúrgica.

## Conclusão Definitiva da Trilha de Regressão Logística
O Experimento 6 encerra esta trilha cumprindo rigorosamente o seu papel no projeto AquaSense.

A Regressão Logística foi levada ao seu limite máximo: recebeu as melhores variáveis (engenharia de atributos), o melhor tratamento de desbalanceamento (`class_weight='balanced'`) e a melhor otimização matemática (Grid Search). O modelo provou ser uma excelente *baseline* — rápida, interpretável e segura do ponto de vista de rastreio de anomalias.

No entanto, a incapacidade de reduzir os falsos alarmes sem destruir o *Recall* atesta que a inteligência final do sistema de monitoramento de Pernambuco deve, obrigatoriamente, ser ancorada em algoritmos baseados em árvores de decisão (como Random Forest e LightGBM), que possuem a arquitetura nativa necessária para lidar com as interações não-lineares da qualidade da água.

In [11]:
# Salvar os resultados para a nossa tabela comparativa geral
resultados = {
    "experimento": "exp_06_logistic_regression_tuning",
    "algoritmo": "LogisticRegression",
    "balanceamento": "class_weight_balanced + tuning",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/EDA_AquaSense/resultados_experimentos.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas finais do Experimento 6 salvas com sucesso.")

Métricas finais do Experimento 6 salvas com sucesso.
